# ( **Allam** **ديرتنا** - **تدريب** **نموذج** )


## **1**. **Install** **Required** **Libraries**

In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00


## **2**. **Load** **the** **Pre**-**trained** **ALLAM** Model

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. تحديد اسم النموذج
model_id = "humain-ai/ALLaM-7B-Instruct-preview"

# 2. إعدادات الضغط 4-bit المخصصة لـ T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. تحميل الـ Tokenizer والنموذج
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# 4. إعداد LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print(" تم تحميل النموذج وتطبيق إعدادات LoRA الشاملة بنجاح!")

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 39,976,960 || all params: 7,040,536,576 || trainable%: 0.5678
 تم تحميل النموذج وتطبيق إعدادات LoRA الشاملة بنجاح!


# **3**. **ALLaM** **Chat** **Template**

In [4]:
import pandas as pd
import glob, os
from datasets import Dataset

# 1. العثور على الملف وقراءته
paths = glob.glob("Deertna_Dataset_CLEAN*.xlsx") or glob.glob("/content/Deertna_Dataset_CLEAN*.xlsx")
if not paths:
    raise FileNotFoundError("لم يتم العثور على ملف Excel، يرجى رفع الملف أولاً!")

file_path = sorted(paths)[-1]
print("الملف المعتمد:", file_path)

df = pd.read_excel(file_path, sheet_name="dataset")
df['instruction'] = df['instruction'].fillna('')
df['response'] = df['response'].fillna('')

# 2. دالة تطبيق Chat Template الخاص بـ ALLaM
def apply_allam_template(row):
    messages = [
        {"role": "user", "content": str(row["instruction"])},
        {"role": "assistant", "content": str(row["response"])}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

df["text"] = df.apply(apply_allam_template, axis=1)

# 3. التقسيم حسب عمود
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "validation"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

# التحويل المباشر لـ Datasets بدون مشاكل PyArrow
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

print(f" تم تجهيز البيانات بنجاح!")
print(f"التدريب (Train): {len(train_dataset)} | التحقق (Validation): {len(val_dataset)} | الاختبار (Test): {len(test_dataset)}")

الملف المعتمد: Deertna_Dataset_CLEAN (1).xlsx
 تم تجهيز البيانات بنجاح!
التدريب (Train): 805 | التحقق (Validation): 219 | الاختبار (Test): 219


# **4**. **ALLaM** **Model**  **Setup** **and** **Training**

In [5]:

import torch
from trl import SFTTrainer, SFTConfig

print("=" * 60)
print("إعداد التدريب")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())



training_args = SFTConfig(
    output_dir="outputs",

    # Dataset
    dataset_text_field="text",
    max_length=256,

    # Batch
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # Training
    num_train_epochs=5,
    learning_rate=2e-4,
    warmup_steps=5,


    fp16=False,
    bf16=False,

    # Logging
    logging_steps=5,

    # Evaluation
    eval_strategy="epoch",

    # Saving
    save_strategy="epoch",

    # Optimizer
    optim="paged_adamw_8bit",

    # Reports
    report_to="none",
)



trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    args=training_args,
)




print("=" * 60)
print(" بدء تدريب ALLaM...")
print("=" * 60)

trainer.train()




print("=" * 60)
print(" حفظ النموذج...")
print("=" * 60)

trainer.save_model("outputs/final_model")
tokenizer.save_pretrained("outputs/final_model")

print("=" * 60)
print(" انتهى التدريب بنجاح!")
print(" outputs/final_model")
print("=" * 60)

إعداد التدريب
GPU: Tesla T4
CUDA available: True


Adding EOS to train dataset:   0%|          | 0/805 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/805 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/805 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/805 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/219 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/219 [00:00<?, ? examples/s]

 بدء تدريب ALLaM...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.301418,0.742773,0.343070,39335.000000,0.880133
2,0.227290,0.719272,0.272238,78670.000000,0.894304
3,0.174906,0.777929,0.228615,118005.000000,0.898565
4,0.159662,0.799883,0.210118,157340.000000,0.899299
5,0.148918,0.819413,0.207335,196675.000000,0.900228


 حفظ النموذج...
 انتهى التدريب بنجاح!
 outputs/final_model


# **5**. **Test** **ALLaM** **Model** **on** **Sample** **Questions**

In [6]:
import torch

def generate_answer(question):
    model.eval()

    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id   # يمنع التحذير
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return answer.strip()


for i in range(5):
    example = test_dataset[i]

    print("=" * 70)
    print("السؤال:")
    print(example["instruction"])

    print("\nالإجابة الصحيحة:")
    print(example["response"])          # كانت output

    print("\nإجابة النموذج:")
    print(generate_answer(example["instruction"]))
    print()

السؤال:
أبغى أعرف معنى «مير»

الإجابة الصحيحة:
كلمة «مير» في لهجة الرياض تعني: لكن / على أية حال.

إجابة النموذج:
كلمة «مير» في لهجة الرياض تعني: لكن / على أية حال.

السؤال:
متى تنقال «مير»؟

الإجابة الصحيحة:
تُستخدم كلمة «مير» في لهجة الرياض كأداة استدراك أو ربط عند الانتقال بين فكرتين في الحديث، أو لتغيير مسار السالفة فجأة.

إجابة النموذج:
تُقال كلمة «مير» في لهجة الرياض كأداة استدراك أو ربط عند الانتقال بين فكرتين في الحديث، أو لتغيير مسار السالفة فجأة.

السؤال:
من أي منطقة كلمة مير وش تعني؟

الإجابة الصحيحة:
كلمة «مير» من لهجة الرياض، ومعناها: لكن / على أية حال.

إجابة النموذج:
كلمة «مير» في لهجة الرياض تعني: لكن / على أية حال.

السؤال:
أبغى أعرف معنى «انثبر»

الإجابة الصحيحة:
كلمة «انثبر» في لهجة الرياض تعني: اجلس واسكت.

إجابة النموذج:
كلمة «انثبر» في لهجة الرياض تعني: اجلس واسكت.

السؤال:
متى تنقال «انثبر»؟

الإجابة الصحيحة:
تُستخدم كلمة «انثبر» في لهجة الرياض كأمر حازم عند الغضب، أو المزاح الثقيل لإسكات الشخص أو أمره بالجلوس دون حركة.

إجابة النموذج:
تُقال كلمة «انثبر» في لهجة 

## **6**. **Load** **the** **Best** **Checkpoint**


In [7]:
print("أفضل checkpoint:", trainer.state.best_model_checkpoint)
print("أفضل eval_loss:", trainer.state.best_metric)
model = trainer.model      # load_best_model_at_end حمّل الأفضل أصلاً
model.eval()

أفضل checkpoint: None
أفضل eval_loss: None


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(64000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## **7**. **Evaluate** **ALLaM** **Model** **Performance**


In [8]:
import re
import torch
from collections import Counter

# 1. دالة تنظيف النص العربي وتوحيده قبل المقارنة
def normalize_text(text):
    text = str(text)
    # إزالة التشكيل
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    # توحيد أشكال الألف والهاء والياء
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    # إزالة علامات الترقيم والرموز
    text = re.sub(r'[^\w\s]', '', text)
    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. دالة حساب F1 Score و Precision و Recall على مستوى الكلمات
def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 0.0, 0.0, 0.0

    # حساب الكلمات المشتركة مع مراعاة التكرار
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0, 0.0, 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

# 3. دالة حساب التطابق التام (Exact Match)
def compute_exact_match(prediction, ground_truth):
    return int(normalize_text(prediction) == normalize_text(ground_truth))

# 4. دالة توليد الإجابة من النموذج
def generate_answer(question):
    model.eval()
    messages = [{"role": "user", "content": question}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,  # Greedy Search للحصول على نتائج دقيقة ومستقرة
            pad_token_id=tokenizer.eos_token_id
        )

    # استخراج النص المولد فقط بدون نص السؤال الأصلي
    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return answer.strip()

# 5. تشغيل التقييم على كامل test_dataset
precisions, recalls, f1s, exact_matches = [], [], [], []

print("=" * 60)
print(" بدء تقييم النموذج على مجموعة الاختبار")
print("=" * 60)

for i, example in enumerate(test_dataset):
    prediction = generate_answer(example["instruction"])
    # التعديل الرئيسي: استخدام response بدلاً من output
    ground_truth = example["response"]

    p, r, f1 = compute_f1(prediction, ground_truth)
    em = compute_exact_match(prediction, ground_truth)

    precisions.append(p)
    recalls.append(r)
    f1s.append(f1)
    exact_matches.append(em)

    print(f"\rتم اختبار {i + 1}/{len(test_dataset)}", end="")

# 6. حساب المتوسطات النهائية
avg_precision = sum(precisions) / len(precisions)
avg_recall = sum(recalls) / len(recalls)
avg_f1 = sum(f1s) / len(f1s)
exact_match_acc = sum(exact_matches) / len(exact_matches)

print("\n")
print("=" * 60)
print(" النتائج النهائية للتقييم")
print("=" * 60)
print(f"عدد أمثلة الاختبار: {len(test_dataset)}")
print(f"متوسط Precision: {avg_precision:.4f}")
print(f"متوسط Recall:    {avg_recall:.4f}")
print(f"متوسط F1 Score:  {avg_f1:.4f}")
print(f"نسبة Exact Match: {exact_match_acc:.2%}")
print("=" * 60)

 بدء تقييم النموذج على مجموعة الاختبار
تم اختبار 219/219

 النتائج النهائية للتقييم
عدد أمثلة الاختبار: 219
متوسط Precision: 0.9395
متوسط Recall:    0.9430
متوسط F1 Score:  0.9408
نسبة Exact Match: 68.49%


## **8**. **Save** **the** **Fine**-**Tuned** **Model**

In [9]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=False)

SAVE_DIR = "/content/drive/MyDrive/Ditratna_ALLAM_v2"
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(" تم حفظ نموذج ديرتنا المدرّب")
print(SAVE_DIR)

MessageError: Error: credential propagation was unsuccessful

# **9**. **Install** **API** **Requirements**

In [10]:
!pip install -q flask flask-cors pyngrok

# **10**. **Create** **Deertna** **Model** **API**

In [11]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.get_json()

        question = data.get("question", "")
        region = data.get("region", "")

        if not question:
            return jsonify({
                "error": "Question is required"
            }), 400

        # إضافة المنطقة كسياق للسؤال
        if region:
            full_question = f"المنطقة: {region}\nالسؤال: {question}"
        else:
            full_question = question

        # استخدام النموذج الذي دربناه
        answer = generate_answer(full_question)

        return jsonify({
            "answer": answer,
            "model": "Deertna_ALLaM"
        })

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500

print("✅ Deertna API is ready")

✅ Deertna API is ready


# **11**. **Run** **the** **Flask** **Server**

In [12]:
import threading
import time

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

time.sleep(2)

print("✅ Deertna Flask server is running on port 5000")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Deertna Flask server is running on port 5000


# **12**. **Test** **the** **API** **Locally**

In [13]:
import requests

test_data = {
    "question": "وش معنى مير؟",
    "region": "الرياض"
}

response = requests.post(
    "http://127.0.0.1:5000/generate",
    json=test_data
)

print("Status:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 00:14:54] "POST /generate HTTP/1.1" 200 -


Status: 200
Response: {'answer': 'كلمة «مير» في لهجة الرياض تعني: لكن / على أية حال.', 'model': 'Deertna_ALLaM'}


# **13**. **Configure** **ngrok**

In [14]:
from pyngrok import ngrok



ngrok.set_auth_token("3JHaBCozAJPlpxkH78CQLPhIED3_27pdrhZDUwXEtCbcruw21")



print("✅ ngrok token added")

✅ ngrok token added


# **14**. **Create** **a** **Public** **API** **Endpoint**

In [15]:
from pyngrok import ngrok

# إغلاق أي tunnel قديم فقط
ngrok.kill()

# فتح رابط خارجي للسيرفر على Port 5000
public_url = ngrok.connect(5000)

print("✅ Deertna ALLaM Public URL:")
print(public_url)
print("\n✅ Generate Endpoint:")
print(f"{public_url}/generate")

✅ Deertna ALLaM Public URL:
NgrokTunnel: "https://washhouse-critter-important.ngrok-free.dev" -> "http://localhost:5000"

✅ Generate Endpoint:
NgrokTunnel: "https://washhouse-critter-important.ngrok-free.dev" -> "http://localhost:5000"/generate


# **15**. **Test** **the** **Public** **API** **Endpoint**

In [16]:
import requests

test_url = public_url.public_url + "/generate"

test_data = {
    "question": "وش معنى مير؟",
    "region": "الرياض"
}

response = requests.post(
    test_url,
    json=test_data
)

print("URL:", test_url)
print("Status:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 00:15:10] "POST /generate HTTP/1.1" 200 -


URL: https://washhouse-critter-important.ngrok-free.dev/generate
Status: 200
Response: {'answer': 'كلمة «مير» في لهجة الرياض تعني: لكن / على أية حال.', 'model': 'Deertna_ALLaM'}


# **16**. **Get** **the** **Final** **API** **Endpoint**

In [17]:
print("انسخي هذا الرابط:")
print(public_url.public_url + "/generate")

انسخي هذا الرابط:
https://washhouse-critter-important.ngrok-free.dev/generate
